# Density-Based Clustering (DBSCAN) — VAST Challenge 2011: Disease Outbreak

This notebook applies **DBSCAN** (Density-Based Spatial Clustering of Applications
with Noise) to geolocated social media messages from the VAST Challenge 2011 dataset,
following the approach described in **Section 6.1** of the paper.

**Scenario:** Citizens of the fictional city of Vastopolis post microblog messages.
A subset of these messages has been labelled as *outbreak-related*. We use DBSCAN to
identify spatial clusters of outbreak activity, then apply the SmartIterator approach
to explore how cluster structure changes as we vary the `eps` (neighbourhood radius)
and `min_samples` parameters.

**Key differences from the K-Means notebook:**
- **Algorithm:** DBSCAN (density-based) instead of K-Means (centroid-based)
- **Features:** Spatial coordinates (X, Y) — longitude and latitude
- **Iteration axis:** Varying `eps` (or `min_samples`) instead of K
- **Noise:** DBSCAN produces a noise class (unassigned points) — absent in K-Means
- **Map:** Scatter plot on the Vastopolis city map (PNG) instead of a choropleth on NUTS3 polygons

**Notebook structure:**
1. Imports & configuration
2. Data loading & filtering
3. Exploratory spatial visualization
4. DBSCAN parameter sweep (compute iterations)
5. Quality metrics across iterations
6. Cluster visualization on map (interactive)
7. 1D Sankey flow diagram
8. Cluster profiles (spatial + temporal)
9. Interactive cluster map explorer
10. Transitions FROM a specific cluster (varying eps)
11. Transitions TO a specific cluster (varying eps)
12. Pair transition between two specific clusters
13. Static HTML export

In [ ]:
# ── Cell 1 — Imports and Configuration ───────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Patch
from matplotlib.colors import to_rgba
from matplotlib.gridspec import GridSpec
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from sklearn.cluster import DBSCAN
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             calinski_harabasz_score, davies_bouldin_score,
                             pairwise_distances)
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import MDS
from scipy.spatial.distance import pdist, squareform
import os
import warnings
warnings.filterwarnings("ignore")

# ── Configuration ──
DATA_DIR = "data\\VastChallenge11"
CSV_FILE = os.path.join(DATA_DIR, "messages_last4days.csv")
MAP_FILE = os.path.join(DATA_DIR, "Vastopolis_Map_grey.png")

# Map bounds (from metadata)
MAP_BOUNDS = {
    'x_min': -93.5673,
    'x_max': -93.1923,
    'y_min': 42.1609,
    'y_max': 42.3017
}

# DBSCAN parameter sweep
EPS_MIN = 0.001
EPS_MAX = 0.010
EPS_STEP = 0.001
MIN_SAMPLES_DEFAULT = 150

# Alternative: fix eps, sweep min_samples
# MIN_SAMPLES_MIN = 3
# MIN_SAMPLES_MAX = 30
# MIN_SAMPLES_STEP = 1

SEED = 42
np.random.seed(SEED)

# Feature columns for clustering
FEATURE_COLS = ['X', 'Y']

print("✅ Configuration loaded")
print(f"   Data:  {CSV_FILE}")
print(f"   Map:   {MAP_FILE}")
print(f"   Eps range: {EPS_MIN:.4f} – {EPS_MAX:.4f} (step {EPS_STEP:.4f})")
print(f"   Min samples: {MIN_SAMPLES_DEFAULT}")
print(f"   → {int((EPS_MAX - EPS_MIN) / EPS_STEP) + 1} iterations planned")

### Cell 2: Load Data and Filter Outbreak-Related Messages

We load the CSV file and retain only messages labelled as **outbreak-related**
(`Outbreak-related (1/0)? == 1`). The clustering features are the spatial
coordinates (X = longitude, Y = latitude).

In [ ]:
# ── Cell 2 — Load and filter data ────────────────────────────────────────────

# Load full dataset
df_raw = pd.read_csv(CSV_FILE)

print(f"Raw dataset: {len(df_raw):,} messages")
print(f"Columns: {list(df_raw.columns)}")

# Filter outbreak-related messages only
df = df_raw[df_raw['Outbreak-related (1/0)?'] == 1].copy().reset_index(drop=True)
print(f"\nOutbreak-related messages: {len(df):,} "
      f"({len(df)/len(df_raw)*100:.1f}% of total)")

# Parse date column
df['datetime'] = pd.to_datetime(df['Created_at'], format='%m/%d/%Y %H:%M')
df['day'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['day_str'] = df['datetime'].dt.strftime('%m/%d')

# Basic stats
print(f"\nDate range: {df['datetime'].min()} → {df['datetime'].max()}")
print(f"Unique days: {df['day'].nunique()}")
print(f"\nSpatial extent:")
print(f"  X (longitude): [{df['X'].min():.5f}, {df['X'].max():.5f}]")
print(f"  Y (latitude):  [{df['Y'].min():.5f}, {df['Y'].max():.5f}]")

# Prepare feature matrix for clustering
data = df[FEATURE_COLS].values
print(f"\nFeature matrix shape: {data.shape}")
print(f"  (each row = one outbreak message with [X, Y] coordinates)")

# Display sample
display(df[['id', 'Name', 'Created_at', 'X', 'Y', 'day_str', 'text']].head(10))

### Cell 3: Exploratory Spatial Visualization

Before clustering, we visualize all outbreak-related messages on the Vastopolis
city map to understand the spatial distribution and identify potential density
patterns that DBSCAN should detect.

In [ ]:
# ── Cell 3 — Exploratory spatial visualization ───────────────────────────────

# Load background map
map_img = mpimg.imread(MAP_FILE)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Left: All outbreak messages on map ──
ax = axes[0]
ax.imshow(map_img, extent=[MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'],
                            MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max']],
          aspect='auto', alpha=0.7, cmap='gray')
ax.scatter(df['X'], df['Y'], s=2, alpha=0.3, c='red', edgecolors='none')
ax.set_xlabel('Longitude (X)')
ax.set_ylabel('Latitude (Y)')
ax.set_title(f'All outbreak-related messages (n={len(df):,})', fontweight='bold')
ax.set_xlim(MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'])
ax.set_ylim(MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max'])

# ── Right: Temporal distribution ──
ax2 = axes[1]
day_counts = df.groupby('day_str').size()
bars = ax2.bar(range(len(day_counts)), day_counts.values, color='coral', edgecolor='darkred')
ax2.set_xticks(range(len(day_counts)))
ax2.set_xticklabels(day_counts.index, rotation=45, ha='right')
ax2.set_xlabel('Date')
ax2.set_ylabel('Number of messages')
ax2.set_title('Outbreak messages per day', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, day_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# ── Hourly pattern ──
fig2, ax3 = plt.subplots(figsize=(10, 3))
hour_counts = df.groupby('hour').size()
ax3.bar(hour_counts.index, hour_counts.values, color='steelblue', edgecolor='navy')
ax3.set_xlabel('Hour of day')
ax3.set_ylabel('Messages')
ax3.set_title('Hourly distribution of outbreak messages', fontweight='bold')
ax3.set_xticks(range(0, 24))
ax3.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Nearest-neighbour distance distribution (for eps selection guidance) ──
# With min_samples=150, we compute distance to the 150th nearest neighbour
# The "elbow" in this plot suggests the optimal eps value.

from sklearn.neighbors import NearestNeighbors

K_NEIGHBORS = MIN_SAMPLES_DEFAULT  # 150
print(f"\nComputing {K_NEIGHBORS}-nearest-neighbour distances...")
print(f"  (this may take a moment for {len(data):,} points)")

nn = NearestNeighbors(n_neighbors=K_NEIGHBORS, algorithm='ball_tree')
nn.fit(data)
distances, _ = nn.kneighbors(data)
k_distances = np.sort(distances[:, -1])  # distance to k-th neighbour

fig3, axes3 = plt.subplots(1, 2, figsize=(16, 5))

# ── Full k-distance plot ──
ax = axes3[0]
ax.plot(k_distances, color='darkblue', linewidth=1)
ax.set_xlabel('Points (sorted by distance)')
ax.set_ylabel(f'Distance to {K_NEIGHBORS}-th nearest neighbour')
ax.set_title(f'k-distance plot (k={K_NEIGHBORS}) — full range', fontweight='bold')
ax.axhline(EPS_MIN, color='green', linestyle='--', alpha=0.7,
           label=f'eps_min={EPS_MIN}')
ax.axhline(EPS_MAX, color='red', linestyle='--', alpha=0.7,
           label=f'eps_max={EPS_MAX}')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Zoomed view: focus on the elbow region ──
ax2 = axes3[1]
# Show only the portion where k-distance is within our eps range
mask_zoom = k_distances <= EPS_MAX * 2
ax2.plot(np.arange(mask_zoom.sum()), k_distances[mask_zoom],
         color='darkblue', linewidth=1.5)
ax2.set_xlabel('Points (sorted, zoomed)')
ax2.set_ylabel(f'Distance to {K_NEIGHBORS}-th nearest neighbour')
ax2.set_title(f'k-distance plot (k={K_NEIGHBORS}) — zoomed to eps range',
              fontweight='bold')
ax2.axhline(EPS_MIN, color='green', linestyle='--', alpha=0.7,
            label=f'eps_min={EPS_MIN}')
ax2.axhline(EPS_MAX, color='red', linestyle='--', alpha=0.7,
            label=f'eps_max={EPS_MAX}')

# Mark percentiles
for pct, style in [(50, ':'), (75, '-.'), (90, '--')]:
    val = np.percentile(k_distances, pct)
    if val <= EPS_MAX * 2:
        ax2.axhline(val, color='purple', linestyle=style, alpha=0.4,
                    label=f'P{pct}={val:.5f}')

ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Statistics to guide eps selection ──
print(f"\n{K_NEIGHBORS}-distance statistics:")
print(f"  Min:    {k_distances[0]:.6f}")
print(f"  P10:    {np.percentile(k_distances, 10):.6f}")
print(f"  P25:    {np.percentile(k_distances, 25):.6f}")
print(f"  Median: {np.median(k_distances):.6f}")
print(f"  P75:    {np.percentile(k_distances, 75):.6f}")
print(f"  P90:    {np.percentile(k_distances, 90):.6f}")
print(f"  P95:    {np.percentile(k_distances, 95):.6f}")
print(f"  P99:    {np.percentile(k_distances, 99):.6f}")
print(f"  Max:    {k_distances[-1]:.6f}")

# Suggest eps range based on elbow
# Points below eps → can be core points; points above → noise
n_below_eps_min = (k_distances <= EPS_MIN).sum()
n_below_eps_max = (k_distances <= EPS_MAX).sum()
print(f"\n  Points with {K_NEIGHBORS}-dist ≤ eps_min ({EPS_MIN}): "
      f"{n_below_eps_min:,} ({n_below_eps_min/len(data)*100:.1f}%)")
print(f"  Points with {K_NEIGHBORS}-dist ≤ eps_max ({EPS_MAX}): "
      f"{n_below_eps_max:,} ({n_below_eps_max/len(data)*100:.1f}%)")
print(f"\n  💡 If most points have {K_NEIGHBORS}-dist > eps_max,")
print(f"     consider increasing EPS_MAX or decreasing MIN_SAMPLES_DEFAULT.")

### Cell 4: Run DBSCAN Parameter Sweep

This cell runs DBSCAN for each value of `eps` in the configured range,
holding `min_samples` constant. For each iteration we record:
- Cluster assignments (including noise label)
- Quality metrics (Silhouette, Calinski-Harabasz, Davies-Bouldin, SSE)
- Medoid (most central point) of each cluster
- Noise count and percentage

This mirrors the K-Means sweep over K, but here the number of clusters
is an *output* of the algorithm (not an input).

In [ ]:
# ── Cell 4 — Run DBSCAN iterations ──────────────────────────────────────────
#
# Two sweep modes supported:
#   1. Fix min_samples, vary eps (default — shows how neighbourhood radius
#      controls granularity)
#   2. Fix eps, vary min_samples (alternative — shows how density threshold
#      controls noise/cluster boundary)
#
# With min_samples=150, only dense concentrations form clusters.
# This is appropriate for the VAST Challenge where outbreak hotspots
# are densely reported areas within the city.

from datetime import datetime

# ── Compute global statistics ──
global_mean = np.mean(data, axis=0)
tss = float(np.sum((data - global_mean) ** 2))

# ── Parameter grid ──
eps_values = np.arange(EPS_MIN, EPS_MAX + EPS_STEP / 2, EPS_STEP)
eps_values = np.round(eps_values, 6)

print(f"Running DBSCAN: {len(eps_values)} iterations")
print(f"  eps: {eps_values[0]:.4f} → {eps_values[-1]:.4f} (step={EPS_STEP:.4f})")
print(f"  min_samples: {MIN_SAMPLES_DEFAULT}")
print(f"  Data points: {len(data):,}")
print(f"  TSS: {tss:.6f}")
print(f"\n  ⚠ With min_samples={MIN_SAMPLES_DEFAULT}, a point needs {MIN_SAMPLES_DEFAULT}")
print(f"    neighbours within eps to be a core point.")
print("=" * 60)

# ── Storage ──
metrics_rows = []
cluster_assignments = {}
centroid_data = {}
iteration_params = {}

for iter_idx, eps_val in enumerate(eps_values):
    iter_num = iter_idx + 1
    t0 = datetime.now()

    # Run DBSCAN
    model = DBSCAN(eps=eps_val, min_samples=MIN_SAMPLES_DEFAULT,
                   algorithm='ball_tree', metric='euclidean')
    raw_labels = model.fit_predict(data)

    # Remap labels: clusters 0..K-1, noise = K
    unique_labels = sorted(set(raw_labels))
    n_clusters = len([l for l in unique_labels if l != -1])

    label_map = {}
    nid = 0
    for l in unique_labels:
        if l == -1:
            continue
        label_map[l] = nid
        nid += 1
    label_map[-1] = nid  # noise gets last ID
    labels = np.array([label_map[l] for l in raw_labels])

    noise_mask = (labels == n_clusters)
    n_noise = int(noise_mask.sum())
    non_noise_mask = ~noise_mask

    # Compute medoids (representative central point per cluster)
    medoids = {}
    for cid in range(n_clusters):
        mask = (labels == cid)
        cdata = data[mask]
        if len(cdata) == 0:
            medoids[cid] = global_mean
        elif len(cdata) <= 5000:
            # True medoid: point with minimum sum of distances
            dm = pairwise_distances(cdata)
            medoids[cid] = cdata[np.argmin(dm.sum(axis=1))]
        else:
            # Approximate: point closest to centroid
            cm = np.mean(cdata, axis=0)
            dists = np.sqrt(((cdata - cm)**2).sum(axis=1))
            medoids[cid] = cdata[np.argmin(dists)]

    # Quality metrics (computed on non-noise points only)
    if n_clusters >= 2 and np.sum(non_noise_mask) > n_clusters:
        sil = silhouette_score(data[non_noise_mask], labels[non_noise_mask])
        calinski = calinski_harabasz_score(data[non_noise_mask], labels[non_noise_mask])
        davies = davies_bouldin_score(data[non_noise_mask], labels[non_noise_mask])
    elif n_clusters == 1 and np.sum(non_noise_mask) > 1:
        sil = 0.0  # undefined for 1 cluster
        calinski = 0.0
        davies = 0.0
    else:
        sil = calinski = davies = 0.0

    # SSE: sum of squared distances to assigned medoid
    sse = 0.0
    if n_clusters > 0:
        for cid in range(n_clusters):
            mask = (labels == cid)
            if mask.sum() > 0:
                sse += float(np.sum((data[mask] - medoids[cid]) ** 2))
    else:
        sse = tss

    var_expl = (1 - sse / tss) * 100 if tss > 0 else 0.0

    # Store
    cluster_assignments[iter_num] = labels.copy()
    iteration_params[iter_num] = {
        'eps': eps_val,
        'min_samples': MIN_SAMPLES_DEFAULT,
        'n_clusters': n_clusters
    }
    centroid_data[iter_num] = {
        'centroids': np.array([medoids[c] for c in range(n_clusters)]) if n_clusters > 0 else np.empty((0, 2)),
        'n_clusters': n_clusters,
        'eps': eps_val
    }

    metrics_rows.append({
        'Iteration': iter_num,
        'Eps': eps_val,
        'Min_Samples': MIN_SAMPLES_DEFAULT,
        'K': n_clusters,
        'N_Noise': n_noise,
        'Noise_Pct': n_noise / len(data) * 100,
        'N_Clustered': int(non_noise_mask.sum()),
        'Clustered_Pct': int(non_noise_mask.sum()) / len(data) * 100,
        'SSE': sse,
        'Variance_Explained': var_expl,
        'Silhouette': sil,
        'Calinski_Harabasz': calinski,
        'Davies_Bouldin': davies
    })

    elapsed = (datetime.now() - t0).total_seconds()
    print(f"  Iter {iter_num:>3}: eps={eps_val:.4f} → "
          f"K={n_clusters:>3}, noise={n_noise:>5} ({n_noise/len(data)*100:.1f}%), "
          f"clustered={int(non_noise_mask.sum()):>5} ({non_noise_mask.sum()/len(data)*100:.1f}%), "
          f"Sil={sil:.3f}  [{elapsed:.2f}s]")

metrics_df = pd.DataFrame(metrics_rows)

print(f"\n{'=' * 60}")
print(f"✅ Done: {len(metrics_df)} iterations (min_samples={MIN_SAMPLES_DEFAULT})")
print(f"   K range: {metrics_df['K'].min()} – {metrics_df['K'].max()}")
print(f"   Noise range: {metrics_df['Noise_Pct'].min():.1f}% – {metrics_df['Noise_Pct'].max():.1f}%")
print(f"   Clustered range: {metrics_df['Clustered_Pct'].min():.1f}% – {metrics_df['Clustered_Pct'].max():.1f}%")

# Iteration range
ITER_MIN = 1
ITER_MAX = len(eps_values)

# ── Quick summary: which iterations have meaningful clusters? ──
meaningful = metrics_df[metrics_df['K'] >= 2]
if len(meaningful) > 0:
    print(f"\n   Iterations with K ≥ 2: {len(meaningful)}")
    print(f"   Eps range for K≥2: {meaningful['Eps'].min():.4f} – {meaningful['Eps'].max():.4f}")
    print(f"   Best Silhouette: {meaningful['Silhouette'].max():.4f} "
          f"at eps={meaningful.loc[meaningful['Silhouette'].idxmax(), 'Eps']:.4f}")
else:
    print("\n   ⚠ No iterations produced K ≥ 2 clusters!")
    print("   → Try increasing EPS_MAX or decreasing MIN_SAMPLES_DEFAULT")

### Cell 4b (Optional): Sweep min_samples at Fixed eps

The paper explores both axes. This cell holds eps fixed at a value identified
from the k-distance plot and varies min_samples to see how the density
threshold affects cluster formation. Higher min_samples → fewer, denser clusters.

In [ ]:
# ── Cell 4b — Optional: sweep min_samples at fixed eps ───────────────────────
# Uncomment and run if you want to explore the min_samples dimension.

SWEEP_MIN_SAMPLES = False  # Set to True to run

if SWEEP_MIN_SAMPLES:
    # Pick eps from k-distance elbow or best iteration
    EPS_FIXED = metrics_df.loc[metrics_df['Silhouette'].idxmax(), 'Eps'] \
                if metrics_df['Silhouette'].max() > 0 else 0.010

    MIN_SAMPLES_VALUES = list(range(20, 301, 10))

    print(f"Sweeping min_samples at fixed eps={EPS_FIXED:.4f}")
    print(f"  min_samples: {MIN_SAMPLES_VALUES[0]} → {MIN_SAMPLES_VALUES[-1]}")
    print("=" * 60)

    ms_rows = []
    for ms in MIN_SAMPLES_VALUES:
        model = DBSCAN(eps=EPS_FIXED, min_samples=ms,
                       algorithm='ball_tree', metric='euclidean')
        raw_labels = model.fit_predict(data)
        n_cl = len(set(raw_labels)) - (1 if -1 in raw_labels else 0)
        n_noise = (raw_labels == -1).sum()
        ms_rows.append({
            'min_samples': ms, 'K': n_cl,
            'N_Noise': n_noise, 'Noise_Pct': n_noise/len(data)*100
        })
        print(f"  min_samples={ms:>4} → K={n_cl:>3}, "
              f"noise={n_noise:>5} ({n_noise/len(data)*100:.1f}%)")

    ms_df = pd.DataFrame(ms_rows)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(ms_df['min_samples'], ms_df['K'], 'o-', color='navy')
    axes[0].set_xlabel('min_samples')
    axes[0].set_ylabel('K (clusters)')
    axes[0].set_title(f'Clusters vs. min_samples (eps={EPS_FIXED:.4f})', fontweight='bold')
    axes[0].axvline(MIN_SAMPLES_DEFAULT, color='red', linestyle='--',
                    label=f'Default={MIN_SAMPLES_DEFAULT}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(ms_df['min_samples'], ms_df['Noise_Pct'], 'o-', color='orange')
    axes[1].set_xlabel('min_samples')
    axes[1].set_ylabel('Noise %')
    axes[1].set_title(f'Noise vs. min_samples (eps={EPS_FIXED:.4f})', fontweight='bold')
    axes[1].axvline(MIN_SAMPLES_DEFAULT, color='red', linestyle='--',
                    label=f'Default={MIN_SAMPLES_DEFAULT}')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

### Cell 5: Quality Metrics Across Iterations

Unlike K-Means (where K is the input and metrics help *choose* K), in DBSCAN
the number of clusters K is an *output*. The iteration axis is `eps` — the
neighbourhood radius.

Key relationships:
- **Small eps** → many small clusters + lots of noise (over-fragmentation)
- **Large eps** → few large clusters + little noise (under-differentiation)
- **Sweet spot** → metrics peak (Silhouette, Calinski-Harabasz) or valley (Davies-Bouldin)

We also plot **K vs. eps** and **noise percentage vs. eps** to understand
the algorithm's sensitivity to the neighbourhood radius.

In [ ]:
# ── Cell 5 — Quality metrics visualization ──────────────────────────────────

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

# ── Plot 1: Number of clusters vs eps ──
ax = axes[0, 0]
ax.plot(metrics_df['Eps'], metrics_df['K'], 'o-', color='navy', markersize=5)
ax.set_xlabel('eps (neighbourhood radius)')
ax.set_ylabel('Number of clusters (K)')
ax.set_title('Clusters Found vs. eps', fontweight='bold')
ax.grid(True, alpha=0.3)

# ── Plot 2: Noise percentage vs eps ──
ax = axes[0, 1]
ax.plot(metrics_df['Eps'], metrics_df['Noise_Pct'], 'o-', color='orange', markersize=5)
ax.set_xlabel('eps')
ax.set_ylabel('Noise %')
ax.set_title('Noise Percentage vs. eps', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(50, color='red', linestyle='--', alpha=0.4, label='50% noise')
ax.legend()

# ── Plot 3: Silhouette score ──
ax = axes[1, 0]
valid = metrics_df[metrics_df['Silhouette'] > 0]
ax.plot(valid['Eps'], valid['Silhouette'], 'o-', color='green', markersize=5)
ax.set_xlabel('eps')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs. eps (↑ better)', fontweight='bold')
ax.grid(True, alpha=0.3)
if len(valid) > 0:
    best_sil = valid.loc[valid['Silhouette'].idxmax()]
    ax.axvline(best_sil['Eps'], color='green', linestyle='--', alpha=0.4)
    ax.annotate(f"best: eps={best_sil['Eps']:.4f}\nSil={best_sil['Silhouette']:.3f}",
                xy=(best_sil['Eps'], best_sil['Silhouette']),
                fontsize=8, color='green',
                textcoords="offset points", xytext=(10, -15))

# ── Plot 4: Variance Explained ──
ax = axes[1, 1]
ax.plot(metrics_df['Eps'], metrics_df['Variance_Explained'], 'o-',
        color='purple', markersize=5)
ax.set_xlabel('eps')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Variance Explained vs. eps', fontweight='bold')
ax.grid(True, alpha=0.3)

# ── Plot 5: Davies-Bouldin ──
ax = axes[2, 0]
valid_db = metrics_df[metrics_df['Davies_Bouldin'] > 0]
ax.plot(valid_db['Eps'], valid_db['Davies_Bouldin'], 'o-', color='red', markersize=5)
ax.set_xlabel('eps')
ax.set_ylabel('Davies-Bouldin Index')
ax.set_title('Davies-Bouldin Index vs. eps (↓ better)', fontweight='bold')
ax.grid(True, alpha=0.3)

# ── Plot 6: Calinski-Harabasz ──
ax = axes[2, 1]
valid_ch = metrics_df[metrics_df['Calinski_Harabasz'] > 0]
ax.plot(valid_ch['Eps'], valid_ch['Calinski_Harabasz'], 'o-',
        color='teal', markersize=5)
ax.set_xlabel('eps')
ax.set_ylabel('Calinski-Harabasz Index')
ax.set_title('Calinski-Harabasz vs. eps (↑ better)', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle(f"DBSCAN Quality Metrics (min_samples={MIN_SAMPLES_DEFAULT})",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Summary table ──
display(HTML("<h4>Metrics Summary</h4>"))
display(metrics_df.style.format({
    'Eps': '{:.4f}', 'Noise_Pct': '{:.1f}%',
    'SSE': '{:.2f}', 'Variance_Explained': '{:.1f}%',
    'Silhouette': '{:.4f}', 'Calinski_Harabasz': '{:.1f}',
    'Davies_Bouldin': '{:.4f}'
}).background_gradient(subset=['Silhouette'], cmap='Greens')
 .background_gradient(subset=['Noise_Pct'], cmap='Oranges'))

### Cell 6: Assign Consistent Colors to Clusters Across Iterations

Since DBSCAN produces different numbers of clusters at different eps values,
we need a consistent colour scheme so the same spatial region keeps a similar
colour as eps changes. We embed all medoids from all iterations into a shared
2D space using MDS, then map positions to colours.

The noise class always receives a fixed colour (light grey).

In [ ]:
# ── Cell 6 — Consistent colour assignment ───────────────────────────────────

from matplotlib.colors import hsv_to_rgb

def assign_colors_to_iterations(centroid_data, iteration_params):
    """
    Embed all medoids from all iterations into a shared 2D space,
    then assign colours based on position.
    """
    # Collect all medoids
    all_medoids = []
    medoid_keys = []  # (iteration, cluster_id)

    for iter_num in sorted(centroid_data.keys()):
        cd = centroid_data[iter_num]
        n_cl = cd['n_clusters']
        if n_cl == 0:
            continue
        for c in range(n_cl):
            all_medoids.append(cd['centroids'][c])
            medoid_keys.append((iter_num, c))

    if len(all_medoids) == 0:
        print("⚠ No clusters found in any iteration")
        return {}

    all_medoids = np.array(all_medoids)

    # 2D embedding (MDS)
    if len(all_medoids) > 2:
        mds = MDS(n_components=2, random_state=SEED, dissimilarity='euclidean',
                  normalized_stress='auto')
        coords_2d = mds.fit_transform(all_medoids)
    else:
        coords_2d = all_medoids[:, :2] if all_medoids.shape[1] >= 2 else all_medoids

    # Normalize to [0, 1]
    for dim in range(2):
        mn, mx = coords_2d[:, dim].min(), coords_2d[:, dim].max()
        if mx > mn:
            coords_2d[:, dim] = (coords_2d[:, dim] - mn) / (mx - mn)
        else:
            coords_2d[:, dim] = 0.5

    # Map 2D position to HSV colour
    color_map = {}  # (iteration, cluster_id) → hex colour
    for i, (iter_num, cid) in enumerate(medoid_keys):
        hue = coords_2d[i, 0]
        sat = 0.5 + 0.4 * coords_2d[i, 1]  # 0.5–0.9
        val = 0.7 + 0.25 * (1 - coords_2d[i, 1])  # 0.7–0.95
        rgb = hsv_to_rgb([hue, sat, val])
        hex_color = '#{:02x}{:02x}{:02x}'.format(
            int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
        color_map[(iter_num, cid)] = hex_color

    # Assign colours back to centroid_data
    NOISE_COLOR = '#D3D3D3'
    for iter_num in sorted(centroid_data.keys()):
        n_cl = centroid_data[iter_num]['n_clusters']
        colors = []
        for c in range(n_cl):
            colors.append(color_map.get((iter_num, c), '#888888'))
        colors.append(NOISE_COLOR)  # noise always grey
        centroid_data[iter_num]['colors'] = colors
        centroid_data[iter_num]['coords_2d'] = np.array([
            coords_2d[medoid_keys.index((iter_num, c))]
            for c in range(n_cl)
        ]) if n_cl > 0 else np.empty((0, 2))

    return color_map

NOISE_COLOR = '#D3D3D3'
color_map = assign_colors_to_iterations(centroid_data, iteration_params)

print(f"✅ Colours assigned to {len(color_map)} medoids across {len(centroid_data)} iterations")
print(f"   Noise colour: {NOISE_COLOR} (always grey)")

### Cell 7: Sankey Flow Diagram — Cluster Evolution Across eps Values

This Sankey diagram shows how cluster membership flows as `eps` increases.
Each column represents one DBSCAN iteration (one eps value). Nodes are clusters
(+ noise). Bands connect clusters across consecutive eps values, with width
proportional to the number of shared points.

Key patterns to look for:
- **Merging bands** (small eps → large eps): small clusters merge into larger ones
- **Stable wide bands**: robust clusters that persist across eps values
- **Noise band shrinking**: points transition from noise to clusters as eps grows

In [ ]:
# ── Cell 7 — Sankey flow diagram ─────────────────────────────────────────────

def build_sankey_transitions(iter_range=None, max_display=15):
    """
    Build a Sankey diagram showing cluster transitions across iterations.
    """
    if iter_range is None:
        iter_range = (ITER_MIN, ITER_MAX)

    iters = list(range(iter_range[0], iter_range[1] + 1))

    # Skip iterations with 0 clusters
    valid_iters = [i for i in iters if iteration_params[i]['n_clusters'] > 0]
    if len(valid_iters) < 2:
        print("⚠ Need at least 2 iterations with clusters")
        return

    # Limit display range
    if len(valid_iters) > max_display:
        step = len(valid_iters) // max_display
        valid_iters = valid_iters[::step]
        if valid_iters[-1] != iters[-1]:
            valid_iters.append(iters[-1])

    # Build node list
    node_labels = []
    node_colors = []
    node_positions = {}  # (iter, cluster) → node_index

    for col_idx, it in enumerate(valid_iters):
        n_cl = iteration_params[it]['n_clusters']
        eps = iteration_params[it]['eps']
        colors = centroid_data[it].get('colors', ['#888888'] * (n_cl + 1))

        for c in range(n_cl + 1):  # +1 for noise
            idx = len(node_labels)
            node_positions[(it, c)] = idx
            if c < n_cl:
                size = int((cluster_assignments[it] == c).sum())
                node_labels.append(f"ε={eps:.3f}<br>C{c} ({size})")
                node_colors.append(colors[c])
            else:
                size = int((cluster_assignments[it] == c).sum())
                node_labels.append(f"ε={eps:.3f}<br>Noise ({size})")
                node_colors.append(NOISE_COLOR)

    # Build links
    sources, targets, values, link_colors = [], [], [], []

    for i in range(len(valid_iters) - 1):
        it1 = valid_iters[i]
        it2 = valid_iters[i + 1]
        labels1 = cluster_assignments[it1]
        labels2 = cluster_assignments[it2]
        n_cl1 = iteration_params[it1]['n_clusters']
        n_cl2 = iteration_params[it2]['n_clusters']
        colors1 = centroid_data[it1].get('colors', ['#888888'] * (n_cl1 + 1))

        for c1 in range(n_cl1 + 1):
            mask1 = (labels1 == c1)
            for c2 in range(n_cl2 + 1):
                mask2 = (labels2 == c2)
                shared = int((mask1 & mask2).sum())
                if shared > 0:
                    sources.append(node_positions[(it1, c1)])
                    targets.append(node_positions[(it2, c2)])
                    values.append(shared)
                    base_color = colors1[c1]
                    r, g, b = int(base_color[1:3], 16), int(base_color[3:5], 16), int(base_color[5:7], 16)
                    link_colors.append(f"rgba({r},{g},{b},0.35)")

    fig = go.Figure(go.Sankey(
        node=dict(pad=15, thickness=20,
                  line=dict(color='black', width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=sources, target=targets,
                  value=values, color=link_colors)
    ))
    fig.update_layout(
        title=f"Cluster Flow: eps={iteration_params[valid_iters[0]]['eps']:.4f} → "
              f"{iteration_params[valid_iters[-1]]['eps']:.4f}",
        height=600, width=1200,
        font_size=9
    )
    fig.show()


build_sankey_transitions()

### Cell 8: Cluster Profiles — Spatial and Temporal Characteristics

For DBSCAN on spatial data, "profiles" describe clusters by:
- **Spatial centroid** (mean X, Y) — where is the cluster located?
- **Spatial spread** (std X, Y) — how compact or dispersed?
- **Temporal distribution** — when do messages in this cluster occur?
- **Cluster size** and density

This is analogous to the Z-score feature profiles in the K-Means notebook,
adapted for the spatial + temporal nature of the VAST Challenge data.

In [ ]:
# ── Cell 8 — Cluster profiles ────────────────────────────────────────────────

def show_cluster_profiles(iter_num):
    """Display spatial and temporal profiles for all clusters at a given iteration."""
    labels = cluster_assignments[iter_num]
    params = iteration_params[iter_num]
    n_cl = params['n_clusters']
    eps = params['eps']
    colors = centroid_data[iter_num].get('colors', ['#888888'] * (n_cl + 1))

    print(f"\n{'═' * 60}")
    print(f"  Cluster Profiles: Iteration {iter_num} (eps={eps:.4f}, K={n_cl})")
    print(f"{'═' * 60}")

    # ── Profile table ──
    profile_rows = []
    for c in range(n_cl + 1):
        mask = (labels == c)
        n_pts = int(mask.sum())
        if n_pts == 0:
            continue
        subset = df.loc[mask]
        is_noise = (c == n_cl)
        profile_rows.append({
            'Cluster': f'Noise' if is_noise else f'C{c}',
            'N': n_pts,
            'Pct': n_pts / len(df) * 100,
            'Mean_X': subset['X'].mean(),
            'Mean_Y': subset['Y'].mean(),
            'Std_X': subset['X'].std(),
            'Std_Y': subset['Y'].std(),
            'Days_active': subset['day'].nunique(),
            'Peak_hour': subset['hour'].mode().iloc[0] if len(subset) > 0 else -1
        })

    profile_df = pd.DataFrame(profile_rows)
    display(profile_df.style.format({
        'Pct': '{:.1f}%', 'Mean_X': '{:.4f}', 'Mean_Y': '{:.4f}',
        'Std_X': '{:.5f}', 'Std_Y': '{:.5f}'
    }))

    # ── Temporal profiles: messages per day per cluster ──
    fig_temp, ax = plt.subplots(figsize=(12, 4))
    days = sorted(df['day_str'].unique())
    x_pos = range(len(days))

    for c in range(n_cl):
        mask = (labels == c)
        day_counts = df.loc[mask].groupby('day_str').size().reindex(days, fill_value=0)
        ax.plot(x_pos, day_counts.values, 'o-', color=colors[c],
                linewidth=2, markersize=5, label=f'C{c} (n={(labels==c).sum()})')

    # Noise
    mask_noise = (labels == n_cl)
    if mask_noise.sum() > 0:
        day_counts_n = df.loc[mask_noise].groupby('day_str').size().reindex(days, fill_value=0)
        ax.plot(x_pos, day_counts_n.values, 's--', color=NOISE_COLOR,
                linewidth=1.5, markersize=4, label=f'Noise (n={mask_noise.sum()})')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(days, rotation=45, ha='right')
    ax.set_xlabel('Date')
    ax.set_ylabel('Messages')
    ax.set_title(f'Temporal profile per cluster (eps={eps:.4f})', fontweight='bold')
    ax.legend(fontsize=8, loc='best', ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ── Hourly profiles ──
    fig_hour, ax2 = plt.subplots(figsize=(12, 4))
    hours = range(24)
    for c in range(min(n_cl, 10)):  # limit to 10 clusters for readability
        mask = (labels == c)
        hour_counts = df.loc[mask].groupby('hour').size().reindex(hours, fill_value=0)
        hour_pct = hour_counts / hour_counts.sum() * 100
        ax2.plot(hours, hour_pct.values, '-', color=colors[c],
                 linewidth=2, alpha=0.8, label=f'C{c}')

    ax2.set_xticks(range(0, 24))
    ax2.set_xlabel('Hour of day')
    ax2.set_ylabel('% of cluster messages')
    ax2.set_title(f'Hourly activity profiles (eps={eps:.4f})', fontweight='bold')
    ax2.legend(fontsize=7, loc='best', ncol=3)
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ── Widget for interactive selection ──
w_iter_profile = widgets.IntSlider(
    value=ITER_MAX // 2, min=ITER_MIN, max=ITER_MAX, step=1,
    description='Iteration:', layout=widgets.Layout(width='400px'))
btn_profile = widgets.Button(description='Show Profiles',
                              button_style='info')
out_profile = widgets.Output()

def on_btn_profile(b):
    with out_profile:
        clear_output(wait=True)
        show_cluster_profiles(w_iter_profile.value)

btn_profile.on_click(on_btn_profile)
display(widgets.HBox([w_iter_profile, btn_profile]))
display(out_profile)

### Cell 9: Interactive Cluster Map Explorer

Displays cluster assignments on the Vastopolis city map background for any
selected iteration. Points are colour-coded by cluster membership; noise
points shown in grey. This is the spatial equivalent of the choropleth map
in the K-Means notebook.

In [ ]:
# ── Cell 9 — Interactive cluster map explorer ────────────────────────────────

def plot_cluster_map(iter_num, point_size=4, show_noise=True, alpha=0.6):
    """Plot clusters on the Vastopolis map for a given iteration."""
    labels = cluster_assignments[iter_num]
    params = iteration_params[iter_num]
    n_cl = params['n_clusters']
    eps = params['eps']
    colors = centroid_data[iter_num].get('colors', ['#888888'] * (n_cl + 1))

    fig, ax = plt.subplots(figsize=(14, 10))

    # Background map
    ax.imshow(map_img, extent=[MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'],
                                MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max']],
              aspect='auto', alpha=0.5, cmap='gray')

    # Plot noise first (background)
    if show_noise:
        noise_mask = (labels == n_cl)
        if noise_mask.sum() > 0:
            ax.scatter(df.loc[noise_mask, 'X'], df.loc[noise_mask, 'Y'],
                       s=point_size * 0.5, c=NOISE_COLOR, alpha=0.3,
                       edgecolors='none', label=f'Noise ({noise_mask.sum():,})')

    # Plot clusters
    for c in range(n_cl):
        mask = (labels == c)
        n_pts = int(mask.sum())
        if n_pts > 0:
            ax.scatter(df.loc[mask, 'X'], df.loc[mask, 'Y'],
                       s=point_size, c=colors[c], alpha=alpha,
                       edgecolors='none', label=f'C{c} ({n_pts:,})')

            # Mark medoid
            medoid = centroid_data[iter_num]['centroids'][c]
            ax.scatter(medoid[0], medoid[1], s=100, c=colors[c],
                       edgecolors='black', linewidths=2, marker='*', zorder=10)

    ax.set_xlim(MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'])
    ax.set_ylim(MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max'])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'DBSCAN Clusters — Iteration {iter_num}: eps={eps:.4f}, '
                 f'K={n_cl}, noise={int((labels==n_cl).sum()):,}',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=7, framealpha=0.9,
              markerscale=3, ncol=2 if n_cl > 8 else 1)
    plt.tight_layout()
    plt.show()


# ── Widget ──
w_iter_map = widgets.IntSlider(
    value=ITER_MAX // 2, min=ITER_MIN, max=ITER_MAX, step=1,
    description='Iteration:', layout=widgets.Layout(width='400px'))
w_ptsize = widgets.FloatSlider(value=5, min=1, max=20, step=1,
                                description='Point size:')
w_noise = widgets.Checkbox(value=True, description='Show noise')
btn_map = widgets.Button(description='Show Map', button_style='primary')
out_map = widgets.Output()

def on_btn_map(b):
    with out_map:
        clear_output(wait=True)
        plot_cluster_map(w_iter_map.value, w_ptsize.value, w_noise.value)

btn_map.on_click(on_btn_map)
display(widgets.HBox([w_iter_map, w_ptsize, w_noise, btn_map]))
display(out_map)

### Cell 10: Transitions FROM a Specific Cluster to a Later Iteration

Given a source cluster at iteration I₁ (specific eps value), this cell tracks
where its members end up at a later iteration I₂ (larger eps). It answers:

> *"As we increase the neighbourhood radius, which clusters absorb this group?"*

**Outputs:**
1. **Summary table** — destination clusters ranked by member count
2. **Sankey diagram** — single source → multiple destinations
3. **Line chart** — spatial profiles (mean X, Y, spread) of each destination sub-group
4. **Radar chart** — temporal profiles of each sub-group
5. **Scatter map** — source cluster members colour-coded by destination

For spatial DBSCAN, the "profiles" are:
- Spatial: mean X, mean Y, std X, std Y (location and compactness)
- Temporal: distribution across days and hours

In [ ]:
# ── Cell 10 — Transitions FROM a specific cluster ────────────────────────────

# Feature names for profile comparison
PROFILE_FEATURES = ['Mean_X', 'Mean_Y', 'Std_X', 'Std_Y',
                    'Day1_pct', 'Day2_pct', 'Day3_pct', 'Day4_pct']

def compute_subgroup_profile(mask):
    """Compute spatial + temporal profile for a subgroup."""
    subset = df.loc[mask]
    if len(subset) == 0:
        return np.zeros(len(PROFILE_FEATURES))

    days = sorted(df['day_str'].unique())
    day_pcts = []
    for d in days[:4]:  # up to 4 days
        day_pcts.append((subset['day_str'] == d).sum() / len(subset) * 100)
    while len(day_pcts) < 4:
        day_pcts.append(0)

    return np.array([
        subset['X'].mean(), subset['Y'].mean(),
        subset['X'].std() if len(subset) > 1 else 0,
        subset['Y'].std() if len(subset) > 1 else 0,
    ] + day_pcts)


def show_transitions_from_cluster(iter1, cluster_id, iter2):
    """Track members of a specific cluster at iter1 to their destinations at iter2."""
    labels1 = cluster_assignments[iter1]
    labels2 = cluster_assignments[iter2]
    params1 = iteration_params[iter1]
    params2 = iteration_params[iter2]
    n_cl2 = params2['n_clusters']

    source_mask = (labels1 == cluster_id)
    n_source = int(source_mask.sum())

    if n_source == 0:
        print(f"⚠ No members in iteration {iter1}, cluster {cluster_id}")
        return

    is_noise_source = (cluster_id == params1['n_clusters'])
    source_label = f"Noise" if is_noise_source else f"C{cluster_id}"

    # Count transitions
    dest_labels = labels2[source_mask]
    dest_clusters, counts = np.unique(dest_labels, return_counts=True)
    sort_idx = np.argsort(-counts)
    dest_clusters = dest_clusters[sort_idx]
    counts = counts[sort_idx]
    pcts = counts / n_source * 100

    # Print summary
    print(f"\n{'═'*65}")
    print(f"  Transitions FROM Iter{iter1} (eps={params1['eps']:.4f}).{source_label} "
          f"({n_source} pts)")
    print(f"  TO Iter{iter2} (eps={params2['eps']:.4f}, K={n_cl2})")
    print(f"{'═'*65}")
    print(f"  {'Destination':<25} {'Count':>7} {'Percent':>9}")
    print(f"  {'─'*45}")
    for dc, cnt, pct in zip(dest_clusters, counts, pcts):
        is_noise_d = (dc == n_cl2)
        lbl = 'Noise' if is_noise_d else f'C{dc}'
        bar = '█' * int(pct / 3)
        print(f"  Iter{iter2}.{lbl:<18} {cnt:>7,} {pct:>7.1f}%  {bar}")

    # Colour palette
    colors2 = centroid_data[iter2].get('colors', ['#888888'] * (n_cl2 + 1))
    palette = [colors2[int(dc)] if int(dc) < len(colors2) else '#888888'
               for dc in dest_clusters]

    # ── Sankey ──
    n_dest = len(dest_clusters)
    node_labels = [f"Iter{iter1}.{source_label}<br>({n_source})"]
    node_colors = ['rgba(70,70,70,0.9)']
    for i, dc in enumerate(dest_clusters):
        is_noise_d = (dc == n_cl2)
        lbl = 'Noise' if is_noise_d else f'C{dc}'
        node_labels.append(f"Iter{iter2}.{lbl}<br>({counts[i]})")
        node_colors.append(palette[i])

    link_colors = []
    for p in palette:
        r, g, b = int(p[1:3], 16), int(p[3:5], 16), int(p[5:7], 16)
        link_colors.append(f"rgba({r},{g},{b},0.4)")

    fig_sankey = go.Figure(go.Sankey(
        node=dict(pad=20, thickness=25, line=dict(color='black', width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=[0]*n_dest, target=list(range(1, n_dest+1)),
                  value=counts.tolist(), color=link_colors)
    ))
    fig_sankey.update_layout(
        title=f"Transitions: Iter{iter1}.{source_label} → Iter{iter2}",
        height=max(350, 50*n_dest), width=700)
    fig_sankey.show()

    # ── Profile comparison (spatial + temporal) ──
    source_profile = compute_subgroup_profile(source_mask)
    dest_profiles = {}
    for dc in dest_clusters:
        sub_mask = source_mask & (labels2 == dc)
        if sub_mask.sum() > 0:
            dest_profiles[dc] = compute_subgroup_profile(sub_mask)

    # Line chart
    fig_line, ax_line = plt.subplots(figsize=(10, 5))
    x_pos = range(len(PROFILE_FEATURES))

    ax_line.plot(x_pos, source_profile, 'k-', linewidth=3, alpha=0.4,
                 label=f'Iter{iter1}.{source_label} (all, n={n_source})', zorder=1)

    for i, dc in enumerate(dest_clusters):
        if dc in dest_profiles:
            is_noise_d = (dc == n_cl2)
            lbl = 'Noise' if is_noise_d else f'C{dc}'
            n_sub = int((source_mask & (labels2 == dc)).sum())
            ax_line.plot(x_pos, dest_profiles[dc], color=palette[i],
                         linewidth=2.5, marker='o', markersize=6,
                         label=f'→ Iter{iter2}.{lbl} (n={n_sub})', zorder=2)

    ax_line.axhline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
    ax_line.set_xticks(x_pos)
    ax_line.set_xticklabels(PROFILE_FEATURES, rotation=35, ha='right', fontsize=9)
    ax_line.set_ylabel('Value')
    ax_line.set_title(f'Profile comparison: Iter{iter1}.{source_label} → Iter{iter2} destinations',
                      fontweight='bold')
    ax_line.legend(fontsize=8, loc='best')
    ax_line.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Radar chart
    n_features = len(PROFILE_FEATURES)
    angles = np.linspace(0, 2*np.pi, n_features, endpoint=False).tolist()
    angles += angles[:1]

    # Normalize profiles for radar (min-max across all displayed profiles)
    all_vals = np.vstack([source_profile] + [dest_profiles[dc] for dc in dest_clusters if dc in dest_profiles])
    v_min = all_vals.min(axis=0)
    v_max = all_vals.max(axis=0)
    v_range = v_max - v_min
    v_range[v_range == 0] = 1

    fig_radar, ax_radar = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    norm_src = ((source_profile - v_min) / v_range).tolist() + [((source_profile - v_min) / v_range)[0]]
    ax_radar.plot(angles, norm_src, 'k-', linewidth=2, alpha=0.4,
                  label=f'Iter{iter1}.{source_label} (all)')
    ax_radar.fill(angles, norm_src, color='grey', alpha=0.05)

    for i, dc in enumerate(dest_clusters):
        if dc in dest_profiles:
            is_noise_d = (dc == n_cl2)
            lbl = 'Noise' if is_noise_d else f'C{dc}'
            norm_d = ((dest_profiles[dc] - v_min) / v_range).tolist()
            norm_d += [norm_d[0]]
            ax_radar.plot(angles, norm_d, color=palette[i], linewidth=2,
                          marker='o', markersize=4, label=f'→ {lbl}')
            ax_radar.fill(angles, norm_d, color=palette[i], alpha=0.08)

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(PROFILE_FEATURES, fontsize=8)
    ax_radar.set_title(f'Radar: Iter{iter1}.{source_label} → Iter{iter2}',
                       fontweight='bold', pad=20)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=7)
    plt.tight_layout()
    plt.show()

    # ── Map: source cluster coloured by destination ──
    fig_map, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(map_img, extent=[MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'],
                                MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max']],
              aspect='auto', alpha=0.4, cmap='gray')

    # Grey background for non-source points
    non_source = ~source_mask
    ax.scatter(df.loc[non_source, 'X'], df.loc[non_source, 'Y'],
               s=1, c='#E0E0E0', alpha=0.15, edgecolors='none')

    for i, dc in enumerate(dest_clusters):
        sub_mask = source_mask & (labels2 == dc)
        n_sub = int(sub_mask.sum())
        if n_sub > 0:
            is_noise_d = (dc == n_cl2)
            lbl = 'Noise' if is_noise_d else f'C{dc}'
            ax.scatter(df.loc[sub_mask, 'X'], df.loc[sub_mask, 'Y'],
                       s=8, c=palette[i], alpha=0.7, edgecolors='none',
                       label=f'→ Iter{iter2}.{lbl} ({n_sub})')

    ax.set_xlim(MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'])
    ax.set_ylim(MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max'])
    ax.set_title(f'Where do members of Iter{iter1}.{source_label} go at Iter{iter2}?',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9, markerscale=3)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# ── Widget ──
w_i1_from = widgets.IntSlider(value=3, min=ITER_MIN, max=ITER_MAX, step=1,
                               description='Source iter:',
                               layout=widgets.Layout(width='300px'))
w_c_from = widgets.IntText(value=0, description='Cluster:',
                            layout=widgets.Layout(width='150px'))
w_i2_from = widgets.IntSlider(value=ITER_MAX//2, min=ITER_MIN, max=ITER_MAX, step=1,
                               description='Target iter:',
                               layout=widgets.Layout(width='300px'))
btn_from = widgets.Button(description='Show Transitions',
                           button_style='info', icon='arrow-right')
out_from = widgets.Output()

def on_btn_from(b):
    with out_from:
        clear_output(wait=True)
        show_transitions_from_cluster(w_i1_from.value, w_c_from.value, w_i2_from.value)

btn_from.on_click(on_btn_from)
display(widgets.HBox([w_i1_from, w_c_from, w_i2_from, btn_from]))
display(out_from)

### Cell 11: Transitions TO a Specific Cluster from an Earlier Iteration

The reverse of Cell 10: given a target cluster at iteration I₂ (larger eps),
trace where its members came from at an earlier iteration I₁ (smaller eps).

> *"What is the composition of this cluster — did it form by merging
> multiple smaller clusters, or is it a direct descendant of one?"*

**Outputs:**
1. Summary table of origin clusters
2. Sankey diagram (multiple origins → single target)
3. Line chart comparing spatial/temporal profiles of origin sub-groups
4. Radar chart for normalized profile comparison
5. Scatter map with origin colour-coding

In [ ]:
# ── Cell 11 — Transitions TO a specific cluster ─────────────────────────────

def show_transitions_to_cluster(iter1, iter2, cluster_id):
    """Trace origins of a specific cluster at iter2 from iter1."""
    labels1 = cluster_assignments[iter1]
    labels2 = cluster_assignments[iter2]
    params1 = iteration_params[iter1]
    params2 = iteration_params[iter2]
    n_cl1 = params1['n_clusters']
    n_cl2 = params2['n_clusters']

    target_mask = (labels2 == cluster_id)
    n_target = int(target_mask.sum())

    if n_target == 0:
        print(f"⚠ No members in iteration {iter2}, cluster {cluster_id}")
        return

    is_noise_target = (cluster_id == n_cl2)
    target_label = "Noise" if is_noise_target else f"C{cluster_id}"

    # Count origins
    origin_labels = labels1[target_mask]
    origin_clusters, counts = np.unique(origin_labels, return_counts=True)
    sort_idx = np.argsort(-counts)
    origin_clusters = origin_clusters[sort_idx]
    counts = counts[sort_idx]
    pcts = counts / n_target * 100

    # Print summary
    print(f"\n{'═'*65}")
    print(f"  Transitions TO Iter{iter2} (eps={params2['eps']:.4f}).{target_label} "
          f"({n_target} pts)")
    print(f"  FROM Iter{iter1} (eps={params1['eps']:.4f}, K={n_cl1})")
    print(f"{'═'*65}")
    print(f"  {'Origin':<25} {'Count':>7} {'Percent':>9}")
    print(f"  {'─'*45}")
    for oc, cnt, pct in zip(origin_clusters, counts, pcts):
        is_noise_o = (oc == n_cl1)
        lbl = 'Noise' if is_noise_o else f'C{oc}'
        bar = '█' * int(pct / 3)
        print(f"  Iter{iter1}.{lbl:<18} {cnt:>7,} {pct:>7.1f}%  {bar}")

    # Colour palette
    colors1 = centroid_data[iter1].get('colors', ['#888888'] * (n_cl1 + 1))
    palette = [colors1[int(oc)] if int(oc) < len(colors1) else '#888888'
               for oc in origin_clusters]

    # ── Sankey ──
    n_orig = len(origin_clusters)
    node_labels = []
    node_colors = []
    for i, oc in enumerate(origin_clusters):
        is_noise_o = (oc == n_cl1)
        lbl = 'Noise' if is_noise_o else f'C{oc}'
        node_labels.append(f"Iter{iter1}.{lbl}<br>({counts[i]})")
        node_colors.append(palette[i])
    node_labels.append(f"Iter{iter2}.{target_label}<br>({n_target})")
    node_colors.append('rgba(70,70,70,0.9)')

    link_colors = []
    for p in palette:
        r, g, b = int(p[1:3], 16), int(p[3:5], 16), int(p[5:7], 16)
        link_colors.append(f"rgba({r},{g},{b},0.4)")

    fig_sankey = go.Figure(go.Sankey(
        node=dict(pad=20, thickness=25, line=dict(color='black', width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=list(range(n_orig)), target=[n_orig]*n_orig,
                  value=counts.tolist(), color=link_colors)
    ))
    fig_sankey.update_layout(
        title=f"Origins: Iter{iter1} → Iter{iter2}.{target_label}",
        height=max(350, 50*n_orig), width=700)
    fig_sankey.show()

    # ── Profile comparison ──
    target_profile = compute_subgroup_profile(target_mask)
    origin_profiles = {}
    for oc in origin_clusters:
        sub_mask = target_mask & (labels1 == oc)
        if sub_mask.sum() > 0:
            origin_profiles[oc] = compute_subgroup_profile(sub_mask)

    # Line chart
    fig_line, ax_line = plt.subplots(figsize=(10, 5))
    x_pos = range(len(PROFILE_FEATURES))

    ax_line.plot(x_pos, target_profile, 'k-', linewidth=3, alpha=0.4,
                 label=f'Iter{iter2}.{target_label} (all, n={n_target})', zorder=1)

    for i, oc in enumerate(origin_clusters):
        if oc in origin_profiles:
            is_noise_o = (oc == n_cl1)
            lbl = 'Noise' if is_noise_o else f'C{oc}'
            n_sub = int((target_mask & (labels1 == oc)).sum())
            ax_line.plot(x_pos, origin_profiles[oc], color=palette[i],
                         linewidth=2.5, marker='o', markersize=6,
                         label=f'← Iter{iter1}.{lbl} (n={n_sub})', zorder=2)

    ax_line.set_xticks(x_pos)
    ax_line.set_xticklabels(PROFILE_FEATURES, rotation=35, ha='right', fontsize=9)
    ax_line.set_ylabel('Value')
    ax_line.set_title(f'Profile comparison: origins → Iter{iter2}.{target_label}',
                      fontweight='bold')
    ax_line.legend(fontsize=8, loc='best')
    ax_line.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Radar chart
    n_features = len(PROFILE_FEATURES)
    angles = np.linspace(0, 2*np.pi, n_features, endpoint=False).tolist()
    angles += angles[:1]

    all_vals = np.vstack([target_profile] + [origin_profiles[oc] for oc in origin_clusters if oc in origin_profiles])
    v_min = all_vals.min(axis=0)
    v_max = all_vals.max(axis=0)
    v_range = v_max - v_min
    v_range[v_range == 0] = 1

    fig_radar, ax_radar = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    norm_tgt = ((target_profile - v_min) / v_range).tolist() + [((target_profile - v_min) / v_range)[0]]
    ax_radar.plot(angles, norm_tgt, 'k-', linewidth=2, alpha=0.4,
                  label=f'Iter{iter2}.{target_label} (all)')
    ax_radar.fill(angles, norm_tgt, color='grey', alpha=0.05)

    for i, oc in enumerate(origin_clusters):
        if oc in origin_profiles:
            is_noise_o = (oc == n_cl1)
            lbl = 'Noise' if is_noise_o else f'C{oc}'
            norm_o = ((origin_profiles[oc] - v_min) / v_range).tolist()
            norm_o += [norm_o[0]]
            ax_radar.plot(angles, norm_o, color=palette[i], linewidth=2,
                          marker='o', markersize=4, label=f'← {lbl}')
            ax_radar.fill(angles, norm_o, color=palette[i], alpha=0.08)

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(PROFILE_FEATURES, fontsize=8)
    ax_radar.set_title(f'Radar: origins → Iter{iter2}.{target_label}',
                       fontweight='bold', pad=20)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=7)
    plt.tight_layout()
    plt.show()

    # ── Map ──
    fig_map, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(map_img, extent=[MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'],
                                MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max']],
              aspect='auto', alpha=0.4, cmap='gray')

    non_target = ~target_mask
    ax.scatter(df.loc[non_target, 'X'], df.loc[non_target, 'Y'],
               s=1, c='#E0E0E0', alpha=0.15, edgecolors='none')

    for i, oc in enumerate(origin_clusters):
        sub_mask = target_mask & (labels1 == oc)
        n_sub = int(sub_mask.sum())
        if n_sub > 0:
            is_noise_o = (oc == n_cl1)
            lbl = 'Noise' if is_noise_o else f'C{oc}'
            ax.scatter(df.loc[sub_mask, 'X'], df.loc[sub_mask, 'Y'],
                       s=8, c=palette[i], alpha=0.7, edgecolors='none',
                       label=f'← Iter{iter1}.{lbl} ({n_sub})')

    ax.set_xlim(MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'])
    ax.set_ylim(MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max'])
    ax.set_title(f'Where do members of Iter{iter2}.{target_label} come from at Iter{iter1}?',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9, markerscale=3)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# ── Widget ──
w_i1_to = widgets.IntSlider(value=3, min=ITER_MIN, max=ITER_MAX, step=1,
                             description='Source iter:',
                             layout=widgets.Layout(width='300px'))
w_i2_to = widgets.IntSlider(value=ITER_MAX//2, min=ITER_MIN, max=ITER_MAX, step=1,
                             description='Target iter:',
                             layout=widgets.Layout(width='300px'))
w_c_to = widgets.IntText(value=0, description='Cluster:',
                          layout=widgets.Layout(width='150px'))
btn_to = widgets.Button(description='Show Origins',
                         button_style='warning', icon='arrow-left')
out_to = widgets.Output()

def on_btn_to(b):
    with out_to:
        clear_output(wait=True)
        show_transitions_to_cluster(w_i1_to.value, w_i2_to.value, w_c_to.value)

btn_to.on_click(on_btn_to)
display(widgets.HBox([w_i1_to, w_i2_to, w_c_to, btn_to]))
display(out_to)

### Cell 12: Pair Transition — Three-Class Decomposition

Given a specific source cluster (Iter₁.C₁) and a specific target cluster (Iter₂.C₂),
perform the three-class decomposition:
- 🟢 **Shared (core)** — in both clusters
- 🔴 **Source only (left behind)** — in C₁ but not C₂
- 🔵 **Target only (newly joined)** — in C₂ but not C₁

Plus profile comparison and ΔProfile bar charts to identify *what distinguishes*
regions that persist vs. those that leave or join.

For spatial DBSCAN this reveals:
- Geographic sub-areas of a cluster that split off at higher eps
- Points at the cluster boundary that get reassigned
- New points previously classified as noise that get absorbed

In [ ]:
# ── Cell 12 — Pair transition (three-class decomposition) ────────────────────

def show_transition_between(iter1, c1, iter2, c2):
    """Three-class decomposition between two specific clusters."""
    labels1 = cluster_assignments[iter1]
    labels2 = cluster_assignments[iter2]
    params1 = iteration_params[iter1]
    params2 = iteration_params[iter2]
    n_cl1 = params1['n_clusters']
    n_cl2 = params2['n_clusters']

    mask_c1 = (labels1 == c1)
    mask_c2 = (labels2 == c2)

    shared = mask_c1 & mask_c2
    source_only = mask_c1 & ~mask_c2
    target_only = ~mask_c1 & mask_c2

    n_c1 = int(mask_c1.sum())
    n_c2 = int(mask_c2.sum())
    n_shared = int(shared.sum())
    n_source_only = int(source_only.sum())
    n_target_only = int(target_only.sum())

    if n_c1 == 0:
        print(f"⚠ No members in iteration {iter1}, cluster {c1}")
        return
    if n_c2 == 0:
        print(f"⚠ No members in iteration {iter2}, cluster {c2}")
        return

    is_noise_1 = (c1 == n_cl1)
    is_noise_2 = (c2 == n_cl2)
    label1 = "Noise" if is_noise_1 else f"C{c1}"
    label2 = "Noise" if is_noise_2 else f"C{c2}"

    jaccard = n_shared / (n_c1 + n_c2 - n_shared) if (n_c1 + n_c2 - n_shared) > 0 else 0
    retention = n_shared / n_c1 * 100 if n_c1 > 0 else 0
    purity = n_shared / n_c2 * 100 if n_c2 > 0 else 0

    # Summary
    print(f"\n{'═'*65}")
    print(f"  Transition: Iter{iter1}.{label1} → Iter{iter2}.{label2}")
    print(f"  (eps: {params1['eps']:.4f} → {params2['eps']:.4f})")
    print(f"{'═'*65}")
    print(f"  Iter{iter1}.{label1} size:  {n_c1:,}")
    print(f"  Iter{iter2}.{label2} size:  {n_c2:,}")
    print(f"  {'─'*40}")
    print(f"  Shared (persistent core):   {n_shared:>6,}  "
          f"({retention:.1f}% of source, {purity:.1f}% of target)")
    print(f"  Source only (left behind):  {n_source_only:>6,}  "
          f"({n_source_only/n_c1*100:.1f}% of source)")
    print(f"  Target only (newly joined): {n_target_only:>6,}  "
          f"({n_target_only/n_c2*100:.1f}% of target)")
    print(f"  {'─'*40}")
    print(f"  Jaccard similarity:         {jaccard:.3f}")
    print(f"  Retention rate:             {retention:.1f}%")
    print(f"  Purity (target):            {purity:.1f}%")

    # Bar chart
    fig_bar = go.Figure()
    categories = ['Source only<br>(left behind)', 'Shared<br>(core)',
                  'Target only<br>(joined)']
    values = [n_source_only, n_shared, n_target_only]
    colors_bar = ['#d62728', '#2ca02c', '#1f77b4']

    fig_bar.add_trace(go.Bar(
        x=categories, y=values, marker_color=colors_bar,
        text=[f"{v:,}<br>({v/max(n_c1,n_c2)*100:.0f}%)" for v in values],
        textposition='outside'
    ))
    fig_bar.update_layout(
        title=f"Iter{iter1}.{label1} → Iter{iter2}.{label2} | Jaccard={jaccard:.3f}",
        yaxis_title="Number of points", height=350, width=550)
    fig_bar.show()

    # ── Profile analysis ──
    class_info = {
        'Shared (core)': {'mask': shared, 'n': n_shared, 'color': '#2ca02c'},
        f'Iter{iter1}.{label1} only': {'mask': source_only, 'n': n_source_only, 'color': '#d62728'},
        f'Iter{iter2}.{label2} only': {'mask': target_only, 'n': n_target_only, 'color': '#1f77b4'},
    }

    profiles = {}
    for label, info in class_info.items():
        if info['n'] > 0:
            profiles[label] = {
                'profile': compute_subgroup_profile(info['mask']),
                'n': info['n'],
                'color': info['color']
            }

    # Line chart
    fig_line, ax_line = plt.subplots(figsize=(11, 5))
    x_pos = range(len(PROFILE_FEATURES))

    for label, pdata in profiles.items():
        ax_line.plot(x_pos, pdata['profile'], color=pdata['color'],
                     linewidth=2.8, marker='o', markersize=7,
                     label=f"{label} (n={pdata['n']})")

    ax_line.axhline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
    ax_line.set_xticks(x_pos)
    ax_line.set_xticklabels(PROFILE_FEATURES, rotation=35, ha='right', fontsize=9)
    ax_line.set_ylabel('Value')
    ax_line.set_title(f'Three-class profiles: Iter{iter1}.{label1} → Iter{iter2}.{label2}',
                      fontweight='bold')
    ax_line.legend(fontsize=8, loc='best')
    ax_line.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Radar chart
    n_features = len(PROFILE_FEATURES)
    angles = np.linspace(0, 2*np.pi, n_features, endpoint=False).tolist()
    angles += angles[:1]

    all_vals = np.vstack([p['profile'] for p in profiles.values()])
    v_min = all_vals.min(axis=0)
    v_max = all_vals.max(axis=0)
    v_range = v_max - v_min
    v_range[v_range == 0] = 1

    fig_radar, ax_radar = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    for label, pdata in profiles.items():
        norm_v = ((pdata['profile'] - v_min) / v_range).tolist()
        norm_v += [norm_v[0]]
        ax_radar.plot(angles, norm_v, color=pdata['color'], linewidth=2.5,
                      marker='o', markersize=4, label=label)
        ax_radar.fill(angles, norm_v, color=pdata['color'], alpha=0.08)

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(PROFILE_FEATURES, fontsize=8)
    ax_radar.set_title(f'Radar: three-class decomposition', fontweight='bold', pad=20)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=7)
    plt.tight_layout()
    plt.show()

    # ── ΔProfile: "left behind" vs core ──
    if n_source_only > 0 and n_shared > 0:
        diff_left = profiles[f'Iter{iter1}.{label1} only']['profile'] - profiles['Shared (core)']['profile']
        fig_diff, ax_diff = plt.subplots(figsize=(10, 4))
        colors_diff = ['#d62728' if v > 0 else '#2ca02c' for v in diff_left]
        ax_diff.bar(x_pos, diff_left, color=colors_diff, alpha=0.8,
                    edgecolor='black', linewidth=0.5)
        ax_diff.axhline(0, color='black', linewidth=0.8)
        ax_diff.set_xticks(x_pos)
        ax_diff.set_xticklabels(PROFILE_FEATURES, rotation=35, ha='right', fontsize=9)
        ax_diff.set_ylabel('Δ Value')
        ax_diff.set_title('What distinguishes "left behind" from "persistent core"?',
                          fontweight='bold')
        ax_diff.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ── ΔProfile: "newly joined" vs core ──
    if n_target_only > 0 and n_shared > 0:
        diff_joined = profiles[f'Iter{iter2}.{label2} only']['profile'] - profiles['Shared (core)']['profile']
        fig_diff2, ax_diff2 = plt.subplots(figsize=(10, 4))
        colors_diff2 = ['#1f77b4' if v > 0 else '#2ca02c' for v in diff_joined]
        ax_diff2.bar(x_pos, diff_joined, color=colors_diff2, alpha=0.8,
                     edgecolor='black', linewidth=0.5)
        ax_diff2.axhline(0, color='black', linewidth=0.8)
        ax_diff2.set_xticks(x_pos)
        ax_diff2.set_xticklabels(PROFILE_FEATURES, rotation=35, ha='right', fontsize=9)
        ax_diff2.set_ylabel('Δ Value')
        ax_diff2.set_title('What distinguishes "newly joined" from "persistent core"?',
                           fontweight='bold')
        ax_diff2.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ── Map: three-class colouring ──
    fig_map, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(map_img, extent=[MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'],
                                MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max']],
              aspect='auto', alpha=0.4, cmap='gray')

    # Grey background
    other_mask = ~(mask_c1 | mask_c2)
    ax.scatter(df.loc[other_mask, 'X'], df.loc[other_mask, 'Y'],
               s=1, c='#E0E0E0', alpha=0.1, edgecolors='none')

    # Three classes
    class_map_info = [
        (source_only, '#d62728', f'Iter{iter1}.{label1} only ({n_source_only})'),
        (shared, '#2ca02c', f'Shared core ({n_shared})'),
        (target_only, '#1f77b4', f'Iter{iter2}.{label2} only ({n_target_only})'),
    ]

    for mask, color, lbl in class_map_info:
        if mask.sum() > 0:
            ax.scatter(df.loc[mask, 'X'], df.loc[mask, 'Y'],
                       s=10, c=color, alpha=0.7, edgecolors='none', label=lbl)

    ax.set_xlim(MAP_BOUNDS['x_min'], MAP_BOUNDS['x_max'])
    ax.set_ylim(MAP_BOUNDS['y_min'], MAP_BOUNDS['y_max'])
    ax.set_title(
        f"Transition: Iter{iter1}.{label1} → Iter{iter2}.{label2}\n"
        f"Green=core | Red=left behind | Blue=newly joined",
        fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9, framealpha=0.9, markerscale=3)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# ── Widget ──
w_i1_pair = widgets.IntSlider(value=3, min=ITER_MIN, max=ITER_MAX, step=1,
                               description='Iter₁:',
                               layout=widgets.Layout(width='250px'))
w_c1_pair = widgets.IntText(value=0, description='C₁:',
                             layout=widgets.Layout(width='120px'))
w_i2_pair = widgets.IntSlider(value=ITER_MAX//2, min=ITER_MIN, max=ITER_MAX, step=1,
                               description='Iter₂:',
                               layout=widgets.Layout(width='250px'))
w_c2_pair = widgets.IntText(value=0, description='C₂:',
                             layout=widgets.Layout(width='120px'))
btn_pair = widgets.Button(description='Show Transition',
                           button_style='success', icon='exchange')
out_pair = widgets.Output()

def on_btn_pair(b):
    with out_pair:
        clear_output(wait=True)
        show_transition_between(w_i1_pair.value, w_c1_pair.value,
                                w_i2_pair.value, w_c2_pair.value)

btn_pair.on_click(on_btn_pair)
display(widgets.HBox([w_i1_pair, w_c1_pair,
                      widgets.Label('  →  '),
                      w_i2_pair, w_c2_pair, btn_pair]))
display(out_pair)

### Cell 13: Static HTML Export

Generates static versions of all key visualizations for HTML export. Since
interactive widgets don't survive `nbconvert`, this cell auto-selects
representative iterations and transitions.

**Included:**
- Quality metrics (6 charts)
- Sankey flow diagram (full eps range)
- Cluster maps for selected iterations
- Transition FROM (largest cluster, early iteration → late iteration)
- Transition TO (largest cluster, late iteration ← early iteration)
- Pair transition (best Jaccard match)
- Most-fragmented cluster example

In [ ]:
# ── Cell 13 — Static HTML export ─────────────────────────────────────────────

from datetime import datetime as dt_now

print(f"{'═' * 70}")
print(f"  STATIC EXPORT — DBSCAN Cluster Analysis")
print(f"  {dt_now.now().strftime('%Y-%m-%d %H:%M')}")
print(f"{'═' * 70}")

# ═══════════════════════════════════════════════════════════════
#  Select representative iterations for export
# ═══════════════════════════════════════════════════════════════

valid_iters = metrics_df[metrics_df['K'] >= 2]
if len(valid_iters) > 0:
    best_sil_iter = int(valid_iters.loc[valid_iters['Silhouette'].idxmax(), 'Iteration'])
else:
    best_sil_iter = ITER_MAX // 2

# Pick evenly spaced iterations
EXPORT_ITERS = sorted(set([
    ITER_MIN,
    max(ITER_MIN, ITER_MAX // 4),
    best_sil_iter,
    max(ITER_MIN, 3 * ITER_MAX // 4),
    ITER_MAX
]))

print(f"\nExport iterations: {EXPORT_ITERS}")
print(f"  (best Silhouette at iteration {best_sil_iter}, "
      f"eps={iteration_params[best_sil_iter]['eps']:.4f})")

# ═══════════════════════════════════════════════════════════════
#  Part 1: Metrics
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 70}")
print("  Part 1: Quality Metrics")
print(f"{'─' * 70}")

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
metric_configs = [
    ('K', 'Number of Clusters vs. eps', 'navy'),
    ('Noise_Pct', 'Noise % vs. eps', 'orange'),
    ('Silhouette', 'Silhouette Score vs. eps', 'green'),
    ('Variance_Explained', 'Variance Explained vs. eps', 'purple'),
    ('Davies_Bouldin', 'Davies-Bouldin vs. eps', 'red'),
    ('Calinski_Harabasz', 'Calinski-Harabasz vs. eps', 'teal'),
]
for i, (col, title, color) in enumerate(metric_configs):
    ax = axes.flat[i]
    valid_m = metrics_df[metrics_df[col] > 0] if col not in ['K', 'Noise_Pct'] else metrics_df
    ax.plot(valid_m['Eps'], valid_m[col], 'o-', color=color, markersize=5)
    ax.set_xlabel('eps')
    ax.set_ylabel(col)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3)
    for ei in EXPORT_ITERS:
        row = metrics_df[metrics_df['Iteration'] == ei]
        if len(row) > 0 and row.iloc[0][col] > 0:
            ax.axvline(row.iloc[0]['Eps'], color=color, linestyle=':', alpha=0.3)

plt.suptitle(f"DBSCAN Metrics (min_samples={MIN_SAMPLES_DEFAULT})",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
#  Part 2: Sankey
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 70}")
print("  Part 2: Sankey Flow Diagram")
print(f"{'─' * 70}")

build_sankey_transitions()

# ═══════════════════════════════════════════════════════════════
#  Part 3: Cluster Maps for selected iterations
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 70}")
print("  Part 3: Cluster Maps")
print(f"{'─' * 70}")

for iter_num in EXPORT_ITERS:
    params = iteration_params[iter_num]
    if params['n_clusters'] == 0:
        print(f"  Iter {iter_num}: 0 clusters, skipping map")
        continue
    plot_cluster_map(iter_num, point_size=5, show_noise=True, alpha=0.6)
    show_cluster_profiles(iter_num)

# ═══════════════════════════════════════════════════════════════
#  Part 4: Transition Analysis
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═' * 70}")
print("  Part 4: CLUSTER TRANSITION ANALYSIS")
print(f"{'═' * 70}")

# Select two iterations for transition analysis
trans_iters = [i for i in EXPORT_ITERS if iteration_params[i]['n_clusters'] >= 2]
if len(trans_iters) >= 2:
    TRANS_I1 = trans_iters[0]
    TRANS_I2 = trans_iters[-1]
else:
    TRANS_I1 = ITER_MIN
    TRANS_I2 = ITER_MAX

print(f"\n  Transition pair: Iter{TRANS_I1} (eps={iteration_params[TRANS_I1]['eps']:.4f}) → "
      f"Iter{TRANS_I2} (eps={iteration_params[TRANS_I2]['eps']:.4f})")

# ── 4a: FROM largest cluster at I1 ──
labels_i1 = cluster_assignments[TRANS_I1]
n_cl_i1 = iteration_params[TRANS_I1]['n_clusters']
cluster_sizes_i1 = [(labels_i1 == c).sum() for c in range(n_cl_i1)]
if len(cluster_sizes_i1) > 0:
    largest_c1 = int(np.argmax(cluster_sizes_i1))
    print(f"\n  4a: FROM Iter{TRANS_I1}.C{largest_c1} "
          f"(n={cluster_sizes_i1[largest_c1]}) → Iter{TRANS_I2}")
    display(HTML(f"<h3>🔀 Transition FROM: Iter{TRANS_I1}.C{largest_c1} → Iter{TRANS_I2}</h3>"))
    show_transitions_from_cluster(TRANS_I1, largest_c1, TRANS_I2)

# ── 4b: TO largest cluster at I2 ──
labels_i2 = cluster_assignments[TRANS_I2]
n_cl_i2 = iteration_params[TRANS_I2]['n_clusters']
cluster_sizes_i2 = [(labels_i2 == c).sum() for c in range(n_cl_i2)]
if len(cluster_sizes_i2) > 0:
    largest_c2 = int(np.argmax(cluster_sizes_i2))
    print(f"\n  4b: TO Iter{TRANS_I2}.C{largest_c2} "
          f"(n={cluster_sizes_i2[largest_c2]}) ← Iter{TRANS_I1}")
    display(HTML(f"<h3>🔀 Transition TO: Iter{TRANS_I1} → Iter{TRANS_I2}.C{largest_c2}</h3>"))
    show_transitions_to_cluster(TRANS_I1, TRANS_I2, largest_c2)

# ── 4c: Best Jaccard pair ──
print(f"\n  4c: Finding best Jaccard pair...")
best_jaccard = -1
best_pair = (0, 0)

for c1 in range(n_cl_i1):
    mask1 = (labels_i1 == c1)
    n1 = int(mask1.sum())
    if n1 == 0:
        continue
    for c2 in range(n_cl_i2):
        mask2 = (labels_i2 == c2)
        n2 = int(mask2.sum())
        if n2 == 0:
            continue
        n_shared = int((mask1 & mask2).sum())
        jac = n_shared / (n1 + n2 - n_shared) if (n1 + n2 - n_shared) > 0 else 0
        if jac > best_jaccard:
            best_jaccard = jac
            best_pair = (c1, c2)

pair_c1, pair_c2 = best_pair
print(f"  Best pair: Iter{TRANS_I1}.C{pair_c1} ↔ Iter{TRANS_I2}.C{pair_c2} "
      f"(Jaccard={best_jaccard:.3f})")
display(HTML(f"<h3>🔀 Pair: Iter{TRANS_I1}.C{pair_c1} → Iter{TRANS_I2}.C{pair_c2} "
             f"(Jaccard={best_jaccard:.3f})</h3>"))
show_transition_between(TRANS_I1, pair_c1, TRANS_I2, pair_c2)

# ── 4d: Most-fragmented cluster ──
print(f"\n  4d: Finding most-fragmented cluster...")
most_split_c = 0
lowest_max_share = 1.0
for c1 in range(n_cl_i1):
    mask1 = (labels_i1 == c1)
    n1 = int(mask1.sum())
    if n1 < 10:
        continue
    dest_labels = labels_i2[mask1]
    _, counts = np.unique(dest_labels, return_counts=True)
    max_share = counts.max() / n1
    if max_share < lowest_max_share:
        lowest_max_share = max_share
        most_split_c = c1

if most_split_c != largest_c1:
    print(f"  Most split: Iter{TRANS_I1}.C{most_split_c} "
          f"(max dest share={lowest_max_share*100:.1f}%)")
    display(HTML(f"<h3>🔀 Most-Split: Iter{TRANS_I1}.C{most_split_c} → Iter{TRANS_I2}</h3>"))
    show_transitions_from_cluster(TRANS_I1, most_split_c, TRANS_I2)

# ═══════════════════════════════════════════════════════════════
#  Summary
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═' * 70}")
print(f"  STATIC EXPORT COMPLETE")
print(f"{'═' * 70}")
print(f"  Data: {len(df):,} outbreak-related messages")
print(f"  Iterations: {ITER_MIN}–{ITER_MAX} (eps={EPS_MIN:.4f}–{EPS_MAX:.4f})")
print(f"  Export iterations: {EXPORT_ITERS}")
print(f"  Transitions: Iter{TRANS_I1} ↔ Iter{TRANS_I2}")
print(f"\n  To export as HTML:")
print(f"    File → Download as → HTML (.html)")
print(f"    or: jupyter nbconvert --to html --execute DBSCAN-VastChallenge11.ipynb")
print(f"{'═' * 70}")